## Step 1: Install Dependencies

In [ ]:
!apt-get install -y tesseract-ocr tesseract-ocr-eng -q
!pip install pytesseract python-docx pillow pandas transformers accelerate torch -q
print('All dependencies installed ✓')

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.0 MB/s eta 0:00:00
All dependencies installed ✓


## Step 2: Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

ROOT        = '/content/drive/MyDrive/'
TRAIN_FOLDER = os.path.join(ROOT, 'train')
TEST_FOLDER  = os.path.join(ROOT, 'test')
TRAIN_CSV    = os.path.join(ROOT, 'train.csv')
TEST_CSV     = os.path.join(ROOT, 'test.csv')

for p in [TRAIN_FOLDER, TEST_FOLDER, TRAIN_CSV, TEST_CSV]:
    status = '✓' if os.path.exists(p) else '✗ NOT FOUND'
    print(f'  {status}  {p}')

Mounted at /content/drive
  ✓  /content/drive/MyDrive/train
  ✓  /content/drive/MyDrive/test
  ✓  /content/drive/MyDrive/train.csv
  ✓  /content/drive/MyDrive/test.csv


## Step 3: Load Local LLM (Qwen2.5-1.5B-Instruct — No API Key)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Loading {MODEL_NAME} ...')
print(f'GPU available: {torch.cuda.is_available()}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    trust_remote_code=True
)
model.eval()
print('Model loaded ✓')

Loading Qwen/Qwen2.5-1.5B-Instruct ...
GPU available: False


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded ✓


## Step 4: Core Utilities (OCR + DOCX Reading + LLM Extraction)

In [ ]:
import json
import re
import glob
import pandas as pd
from pathlib import Path
from docx import Document
from PIL import Image
import pytesseract

# ─────────────────────────────────────────────────────────
# TEXT EXTRACTION
# ─────────────────────────────────────────────────────────

def read_docx(path: str) -> str:
    """Extract text from a .docx file."""
    doc = Document(path)
    lines = []
    for para in doc.paragraphs:
        t = para.text.strip()
        if t:
            lines.append(t)
    # Also extract text from tables
    for table in doc.tables:
        for row in table.rows:
            row_text = ' | '.join(cell.text.strip() for cell in row.cells if cell.text.strip())
            if row_text:
                lines.append(row_text)
    return '\n'.join(lines)


def read_image_ocr(path: str) -> str:
    """Extract text from image using Tesseract OCR."""
    img = Image.open(path)
    # Upscale small images for better OCR accuracy
    w, h = img.size
    if max(w, h) < 1500:
        scale = 1500 / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    # Convert to grayscale for better OCR
    img = img.convert('L')
    text = pytesseract.image_to_string(img, lang='eng', config='--psm 3')
    return text.strip()


def get_document_text(file_path: str) -> str:
    """Auto-detect file type and extract text."""
    ext = Path(file_path).suffix.lower()
    if ext == '.docx':
        return read_docx(file_path)
    elif ext in ('.png', '.jpg', '.jpeg', '.tiff', '.bmp'):
        return read_image_ocr(file_path)
    else:
        raise ValueError(f'Unsupported file type: {ext}')


# ─────────────────────────────────────────────────────────
# LLM-BASED METADATA EXTRACTION
# ─────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a specialist at reading rental and lease agreements.
Extract exactly these 6 fields from the agreement text and return ONLY a valid JSON object.
Do NOT include any explanation, markdown, or extra text — only the JSON.

Rules:
- "Aggrement Value": monthly rent as integer number only (no currency symbols, no commas). Example: 12000
- "Aggrement Start Date": format DD.MM.YYYY. Example: "01.04.2008"
- "Aggrement End Date": format DD.MM.YYYY. Calculate end date from duration if needed. Example: "31.03.2009"
- "Renewal Notice (Days)": integer number of days notice required to terminate/renew.
  Convert text to days: "one month" or "1 month" = 30, "two months" = 60, "three months" = 90,
  "15 days" = 15, "one week" = 7. If not mentioned, use null.
- "Party One": the LESSOR / OWNER / HOUSE OWNER (landlord, first party). Full name only.
- "Party Two": the LESSEE / TENANT / RESIDENT (second party). Full name only.
- Use null for any field that cannot be determined.

Return exactly this structure:
{"Aggrement Value": <integer or null>, "Aggrement Start Date": "<DD.MM.YYYY or null>", "Aggrement End Date": "<DD.MM.YYYY or null>", "Renewal Notice (Days)": <integer or null>, "Party One": "<string or null>", "Party Two": "<string or null>"}"""


def build_prompt(document_text: str) -> str:
    """Build the chat prompt for the LLM."""
    # Truncate very long documents to fit model context (keep first 3000 chars)
    truncated = document_text[:3000]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Extract metadata from this rental agreement:\n\n{truncated}"}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def llm_extract(document_text: str) -> dict:
    """Run LLM inference and return parsed JSON dict."""
    prompt = build_prompt(document_text)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,          # greedy — deterministic, more reliable for JSON
            temperature=None,
            top_p=None,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return parse_json(raw)


def parse_json(raw: str) -> dict:
    """Robustly parse JSON from model output."""
    # Strip markdown fences
    raw = re.sub(r'```(?:json)?\s*', '', raw).strip()
    raw = raw.replace('```', '').strip()
    # Try direct parse
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    # Try extracting first {...} block
    m = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    # Return empty dict — all fields will be null
    print(f'    ⚠️  Could not parse JSON from: {raw[:200]}')
    return {}


# ─────────────────────────────────────────────────────────
# FILE FINDER
# ─────────────────────────────────────────────────────────

def find_file(folder: str, stem: str) -> str | None:
    """Find a file by stem (no extension) in folder, checking docx then images."""
    for ext in ('.docx', '.png', '.jpg', '.jpeg', '.tiff'):
        path = os.path.join(folder, stem + ext)
        if os.path.exists(path):
            return path
    # Case-insensitive glob fallback
    matches = glob.glob(os.path.join(folder, stem + '.*'))
    return matches[0] if matches else None


FIELDS = [
    'Aggrement Value',
    'Aggrement Start Date',
    'Aggrement End Date',
    'Renewal Notice (Days)',
    'Party One',
    'Party Two'
]

print('All utilities loaded ✓')

All utilities loaded ✓


## Step 5: Main Extraction Function

In [ ]:
def extract_metadata(file_path: str) -> dict:
    """
    Full pipeline: file → text → LLM → structured dict.
    Returns dict with 6 metadata fields.
    """
    text = get_document_text(file_path)
    if not text.strip():
        print('    ⚠️  Empty text extracted from file!')
        return {f: None for f in FIELDS}
    result = llm_extract(text)
    # Ensure all expected fields exist
    for f in FIELDS:
        result.setdefault(f, None)
    return result


def normalize(val):
    """Normalize value for recall comparison."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    return str(val).strip().lower().strip(',')


print('extract_metadata() ready ✓')

extract_metadata() ready ✓


## Step 6: Evaluate on Training Set
This runs inference on every file in `train/` and computes per-field recall vs `train.csv`.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
train_df.columns = train_df.columns.str.strip()
print(f'Training samples: {len(train_df)}')
display(train_df)

Training samples: 10


,File Name,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO
1,6683129-House-Rental-Contract-Geraldine-Galina...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO
2,18325926-Rental-Agreement-1,4000,05.12.2008,31.11.2009,90.0,MR.K.Kuttan,P.M. Narayana Namboodri
3,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj
4,36199312-Rental-Agreement,3800,01.05.2010,31.04.2011,30.0,Balaji.R,Kartheek R
5,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,3000,20.09.2010,19.07.2011,NaN,M.V.V. VIJAYA SHANKAR,MADDIREDDY BHARGAVA REDDY
6,47854715-RENTAL-AGREEMENT,9000,01.04.2010,31.02.2011,60.0,P C MATHEW,L GOPINATH
7,50070534-RENTAL-AGREEMENT,10000,01.04.2010,30.03.2011,90.0,P. JohnsonRavikumar,Saravanan BV
8,54770958-Rental-Agreement,8000,01.04.2011,31.03.2012,90.0,K. Parthasarathy,Veerabrahmam Bathini
9,54945838-Rental-Agreement,5500,21.04.2011,19.02.2012,60.0,Asha Ramesh & Ramesh K.N,Sadasivuni Deepthi & Sadasivuni Kiran


In [ ]:
train_records = []

for idx, row in train_df.iterrows():
    stem = str(row['File Name']).strip()
    file_path = find_file(TRAIN_FOLDER, stem)

    print(f'[{idx+1}/{len(train_df)}] {stem}')

    if file_path is None:
        print(f'  ⚠️  File not found in train/ folder')
        rec = {'File Name': stem}
        for f in FIELDS:
            rec[f'pred_{f}'] = None
            rec[f'gt_{f}']   = row.get(f)
            rec[f'match_{f}'] = False
        train_records.append(rec)
        continue

    print(f'  File: {os.path.basename(file_path)}')
    try:
        extracted = extract_metadata(file_path)
    except Exception as e:
        print(f'  ❌ Error: {e}')
        extracted = {f: None for f in FIELDS}

    rec = {'File Name': stem}
    field_matches = []
    for f in FIELDS:
        pred = normalize(extracted.get(f))
        gt   = normalize(row.get(f))
        match = (pred == gt) or (pred is None and gt is None)
        rec[f'pred_{f}'] = extracted.get(f)
        rec[f'gt_{f}']   = row.get(f)
        rec[f'match_{f}'] = match
        field_matches.append(match)
        icon = '✓' if match else '✗'
        print(f'  {icon} {f:35s} pred={extracted.get(f)}  |  gt={row.get(f)}')

    print(f'  → {sum(field_matches)}/{len(FIELDS)} fields correct\n')
    train_records.append(rec)

train_results_df = pd.DataFrame(train_records)
print('\nTraining evaluation complete ✓')

[1/10] 6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1
  File: 6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx
  ✗ Aggrement Value                     pred=6500001  |  gt=6500
  ✓ Aggrement Start Date                pred=20.05.2007  |  gt=20.05.2007
  ✓ Aggrement End Date                  pred=20.05.2008  |  gt=20.05.2008
  ✗ Renewal Notice (Days)               pred=90  |  gt=15.0
  ✗ Party One                           pred=Antonio Levy S. Ingles. Jr. and/or Mary Rose C. Ingles  |  gt=Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles
  ✗ Party Two                           pred=GERALDINE O. GALINATO  |  gt=GERALDINE Q. GALINATO
  → 2/6 fields correct

[2/10] 6683129-House-Rental-Contract-Geraldine-Galinato-v2
  File: 6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx
  ✗ Aggrement Value                     pred=650000  |  gt=6500
  ✓ Aggrement Start Date                pred=20.05.2007  |  gt=20.05.2007
  ✓ Aggrement End Date                  pred=

In [ ]:
print('\n=========================================')
print('   PER-FIELD RECALL — TRAINING SET')
print('=========================================')

recall_scores = {}
for f in FIELDS:
    col = f'match_{f}'
    if col in train_results_df.columns:
        true_count = train_results_df[col].sum()
        total      = len(train_results_df)
        recall     = true_count / total
        recall_scores[f] = recall
        print(f'  {f:35s}: {recall:.2%}  ({int(true_count)}/{total})')

overall = sum(recall_scores.values()) / len(recall_scores)
print(f'\n  Overall average recall: {overall:.2%}')
print('=========================================')


   PER-FIELD RECALL — TRAINING SET
  Aggrement Value                    : 20.00%  (2/10)
  Aggrement Start Date               : 50.00%  (5/10)
  Aggrement End Date                 : 30.00%  (3/10)
  Renewal Notice (Days)              : 10.00%  (1/10)
  Party One                          : 10.00%  (1/10)
  Party Two                          : 20.00%  (2/10)

  Overall average recall: 23.33%


## Step 7: Predict on Test Set

In [ ]:
test_df = pd.read_csv(TEST_CSV)
test_df.columns = test_df.columns.str.strip()
print(f'Test samples: {len(test_df)}')
display(test_df)

Test samples: 4


,File Name,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two
0,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60,Hanumaiah,Vishal Bhardwaj
1,95980236-Rental-Agreement,9000,01.04.2010,31.03.2011,30,S.Sakunthala,V.V.Ravi Kian
2,156155545-Rental-Agreement-Kns-Home,12000,15.12.2012,14.11.2013,30,V.K.NATARAJ,VYSHNAVI DAIRY SPECIALITIES Private Ltd
3,228094620-Rental-Agreement,15000,07.07.2013,06.06.2014,30,KAPIL MEHROTRA,.B.Kishore


In [10]:
test_records = []

for idx, row in test_df.iterrows():
    stem = str(row['File Name']).strip()
    file_path = find_file(TEST_FOLDER, stem)

    print(f'[{idx+1}/{len(test_df)}] {stem}')

    if file_path is None:
        print(f'  ⚠️  File not found in test/ folder')
        test_records.append({'File Name': stem, **{f: None for f in FIELDS}})
        continue

    print(f'  File: {os.path.basename(file_path)}')
    try:
        extracted = extract_metadata(file_path)
        print(f'  ✓ Extracted: {extracted}')
    except Exception as e:
        print(f'  ❌ Error: {e}')
        extracted = {f: None for f in FIELDS}

    test_records.append({'File Name': stem, **{f: extracted.get(f) for f in FIELDS}})
    print()

test_predictions_df = pd.DataFrame(test_records)
print('\nTest predictions complete ✓')
display(test_predictions_df)

[1/4] 24158401-Rental-Agreement
  File: 24158401-Rental-Agreement.png
  ✓ Extracted: {'Aggrement Value': 12000, 'Aggrement Start Date': '01.04.2008', 'Aggrement End Date': '31.03.2009', 'Renewal Notice (Days)': 90, 'Party One': 'Sri Hanumaiah', 'Party Two': 'Sri Vishal Bhardwaj S/o Charnel Singh Village Pandol Road PO and Tehsil Baijnath Dist: Kangra (H.P.) Himachal Pradesh 176125'}

[2/4] 95980236-Rental-Agreement
  File: 95980236-Rental-Agreement.png
  ✓ Extracted: {'Aggrement Value': None, 'Aggrement Start Date': '01.04.2010', 'Aggrement End Date': '31.03.2011', 'Renewal Notice (Days)': 30, 'Party One': 'Mrs. S.Sakunthala', 'Party Two': 'V.V Ravi Kian'}

[3/4] 156155545-Rental-Agreement-Kns-Home
  File: 156155545-Rental-Agreement-Kns-Home.pdf.docx
  ✓ Extracted: {'Aggrement Value': 12000, 'Aggrement Start Date': '01.12.2012', 'Aggrement End Date': '31.12.2015', 'Renewal Notice (Days)': 90, 'Party One': 'V.K.NATARAJ', 'Party Two': 'SRI VYSHNAVI DAIRY SPECIALITIES Private Ltd.'}

[4/4

,File Name,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two
0,24158401-Rental-Agreement,12000.0,01.04.2008,31.03.2009,90,Sri Hanumaiah,Sri Vishal Bhardwaj S/o Charnel Singh Village ...
1,95980236-Rental-Agreement,NaN,01.04.2010,31.03.2011,30,Mrs. S.Sakunthala,V.V Ravi Kian
2,156155545-Rental-Agreement-Kns-Home,12000.0,01.12.2012,31.12.2015,90,V.K.NATARAJ,SRI VYSHNAVI DAIRY SPECIALITIES Private Ltd.
3,228094620-Rental-Agreement,15000.0,07.07.2013,06.06.2014,90,KAPIL MEHROTRA,B.KISHORE


In [11]:
# ── Recall on test set (using test.csv ground truth) ──
print('\n=========================================')
print('   PER-FIELD RECALL — TEST SET')
print('=========================================')

for f in FIELDS:
    true_count = 0
    total = len(test_df)
    for _, row in test_df.iterrows():
        stem     = str(row['File Name']).strip()
        pred_row = test_predictions_df[test_predictions_df['File Name'] == stem]
        if pred_row.empty:
            continue
        gt   = normalize(row.get(f))
        pred = normalize(pred_row.iloc[0].get(f))
        if gt == pred:
            true_count += 1
    recall = true_count / total if total else 0
    print(f'  {f:35s}: {recall:.2%}  ({true_count}/{total})')

print('=========================================')


   PER-FIELD RECALL — TEST SET
  Aggrement Value                    : 0.00%  (0/4)
  Aggrement Start Date               : 75.00%  (3/4)
  Aggrement End Date                 : 75.00%  (3/4)
  Renewal Notice (Days)              : 25.00%  (1/4)
  Party One                          : 50.00%  (2/4)
  Party Two                          : 0.00%  (0/4)


## Step 8: Save Predictions CSV to Google Drive

In [12]:
output_path = os.path.join(ROOT, 'test_predictions.csv')
test_predictions_df.to_csv(output_path, index=False)
print(f'Saved → {output_path}')
display(test_predictions_df)

Saved → /content/drive/MyDrive/test_predictions.csv


,File Name,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two
0,24158401-Rental-Agreement,12000.0,01.04.2008,31.03.2009,90,Sri Hanumaiah,Sri Vishal Bhardwaj S/o Charnel Singh Village ...
1,95980236-Rental-Agreement,NaN,01.04.2010,31.03.2011,30,Mrs. S.Sakunthala,V.V Ravi Kian
2,156155545-Rental-Agreement-Kns-Home,12000.0,01.12.2012,31.12.2015,90,V.K.NATARAJ,SRI VYSHNAVI DAIRY SPECIALITIES Private Ltd.
3,228094620-Rental-Agreement,15000.0,07.07.2013,06.06.2014,90,KAPIL MEHROTRA,B.KISHORE


## Step 9 (Optional): Flask REST API
Wraps the extraction as a local web service. No API key needed — uses the same local model.

In [13]:
!pip install flask -q

import threading, tempfile
from flask import Flask, request, jsonify

flask_app = Flask(__name__)


@flask_app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'Qwen2.5-1.5B-Instruct (local)'})


@flask_app.route('/extract', methods=['POST'])
def extract_api():
    """
    POST /extract
    Form-data: file=<.docx or .png file>
    Returns: JSON with 6 metadata fields
    """
    if 'file' not in request.files:
        return jsonify({'error': "No file provided. Use form-data key 'file'."}), 400

    uploaded = request.files['file']
    ext = Path(uploaded.filename).suffix.lower()

    if ext not in ('.docx', '.png', '.jpg', '.jpeg'):
        return jsonify({'error': f'Unsupported file type: {ext}'}), 400

    with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
        uploaded.save(tmp.name)
        tmp_path = tmp.name

    try:
        result = extract_metadata(tmp_path)
    except Exception as e:
        os.unlink(tmp_path)
        return jsonify({'error': str(e)}), 500

    os.unlink(tmp_path)
    return jsonify(result)


def run_flask():
    flask_app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()

import time; time.sleep(2)
print('Flask API running on http://localhost:5000')
print('  GET  /health')
print('  POST /extract   (form-data: file=<your_file>)')

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Flask API running on http://localhost:5000
  GET  /health
  POST /extract   (form-data: file=<your_file>)


In [14]:
# ── Quick API test ──
import requests as req

# Health check
r = req.get('http://localhost:5000/health')
print('Health:', r.json())

# Test with first file from test set
first_stem = str(test_df.iloc[0]['File Name']).strip()
first_file = find_file(TEST_FOLDER, first_stem)

if first_file:
    with open(first_file, 'rb') as f:
        resp = req.post('http://localhost:5000/extract',
                        files={'file': (os.path.basename(first_file), f)})
    print(f'\nExtraction for {first_stem}:')
    print(json.dumps(resp.json(), indent=2))
else:
    print('Test file not found for API demo.')

INFO:werkzeug:127.0.0.1 - - [27/Jun/2026 09:49:32] "GET /health HTTP/1.1" 200 -


Health: {'model': 'Qwen2.5-1.5B-Instruct (local)', 'status': 'ok'}


INFO:werkzeug:127.0.0.1 - - [27/Jun/2026 09:52:11] "POST /extract HTTP/1.1" 200 -



Extraction for 24158401-Rental-Agreement:
{
  "Aggrement End Date": "31.03.2009",
  "Aggrement Start Date": "01.04.2008",
  "Aggrement Value": 12000,
  "Party One": "Sri Hanumaiah",
  "Party Two": "Sri Vishal Bhardwaj S/o Charnel Singh Village Pandol Road PO and Tehsil Baijnath Dist: Kangra (H.P.) Himachal Pradesh 176125",
  "Renewal Notice (Days)": 90
}
